In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from brochure_pdf import save_brochure_pdf
import gradio as gr

In [2]:
# testing my scaper function working

url = "https://cnn.com"
print(fetch_website_links(url))
print(fetch_website_contents(url))

['https://edition.cnn.com', 'https://edition.cnn.com/us', 'https://edition.cnn.com/world', 'https://edition.cnn.com/politics', 'https://edition.cnn.com/business', 'https://edition.cnn.com/health', 'https://edition.cnn.com/entertainment', 'https://edition.cnn.com/style', 'https://edition.cnn.com/travel', 'https://edition.cnn.com/sports', 'https://edition.cnn.com/science', 'https://edition.cnn.com/climate', 'https://edition.cnn.com/weather', 'https://edition.cnn.com/world/europe/ukraine', 'https://edition.cnn.com/world/middleeast/israel', 'https://edition.cnn.com/games', 'https://edition.cnn.com/watch', 'https://edition.cnn.com/audio', 'https://edition.cnn.com/live-tv', 'https://cnn.com/account/settings', 'https://cnn.com/newsletters', 'https://cnn.com/follow?iid=fw_var-nav', 'https://us.cnn.com?hpt=header_edition-picker', 'https://edition.cnn.com?hpt=header_edition-picker', 'https://arabic.cnn.com?hpt=header_edition-picker', 'https://cnnespanol.cnn.com/?hpt=header_edition-picker', 'http

In [2]:
# Load environment variables in a file called .env (check if using open ai api keys)

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')


# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("gsk"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
groq = OpenAI(base_url="https://api.groq.com/openai/v1",api_key=api_key)
MODEL = "openai/gpt-oss-20b"

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

        """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://cnn.com"))

In [ ]:

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links =json.loads(result)
    return links

In [ ]:
select_relevant_links("https://szn2.in")

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url) # returned with LLM reasoning
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        #print(f'check: {link["url"]}')
        result += fetch_website_contents(link["url"]) # add the contents from the selected relevant links from the page
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://szn2.in"))

In [8]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url) # user prompt stiched from previous LLM call
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("SZN2", "https://szn2.in")

In [24]:
from brochure_pdf import save_brochure_pdf


def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    pdf_path = save_brochure_pdf(result, company_name)
    print(f"Brochure PDF saved to {pdf_path}")
    return result

In [ ]:
save_brochure_pdf(create_brochure("Apple", "https://apple.com"),"Apple")


In [27]:
from brochure_pdf import save_brochure_pdf


def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result
   #pdf_path = save_brochure_pdf(result, company_name)
    #print(f"Brochure PDF saved to {pdf_path}")
    #return pdf_path

In [ ]:
#create_brochure("SZN2", "https://szn2.in")
save_brochure_pdf(create_brochure("SZN2", "https://szn2.in"), "SZN2")


In [ ]:
# Showcasing the same functionality with the help of gradio
company_name = gr.Textbox(label="Company Name", info="Enter a the name of the company you want to create a brochure for", lines=7)
url = gr.Textbox(label="Company URL", info="Enter a the url of the company you want to create a brochure for", lines=7)
message_output = gr.Markdown(label="Response:")
#pdf_path = gr.File(label="PDF Output", file_count="single")
view = gr.Interface(
    fn=create_brochure,
    inputs=[company_name, url],
    outputs=[message_output],
    title="Company Brochure Generator",
    description="Enter a company name and url to generate a brochure",
    flagging_mode="never"
)
view.launch(inbrowser=True)

In [ ]:
#Additonal function to see how streaming works via api call
def stream_brochure(company_name, url):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    # For Typewriter animation as shown in chatgpt UI
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("SZN2", "https://szn2.in")